# Introduction to Bayesian Count Models with PyMCThis notebook demonstrates Bayesian modeling for **count data** - discrete, non-negative integers representing the number of times an event occurs. We'll explore how to:1. **Understand count distributions**: Poisson and Negative Binomial2. **Model simple counts**: Estimate event rates from data3. **Build regression models**: Relate counts to predictor variables4. **Handle overdispersion**: When variance exceeds the mean5. **Compare models**: Use Bayesian model comparison tools6. **Validate predictions**: Posterior predictive checks## What are Count Models?Count models are used when your outcome variable represents **discrete counts**:- Number of disease cases per day- Customer arrivals per hour- Website visits per session- Sequencing reads per gene- Goals scored in a matchUnlike continuous data (which can take any value), counts are:- **Non-negative integers**: 0, 1, 2, 3, ...- **Discrete**: No fractional values (you can't have 2.5 customers)- **Often right-skewed**: Many zeros/small values, few large values## Why Bayesian Count Models?The Bayesian approach provides:- **Uncertainty quantification**: Full posterior distributions for rates and effects- **Flexible modeling**: Easy to add complexity (hierarchical structures, time trends)- **Interpretable results**: Probability statements about parameters- **Overdispersion handling**: Models like Negative Binomial naturally handle variance > mean## Models We'll Cover1. **Poisson Model** (Section 1)   - Simplest count model   - Assumes mean = variance   - Best for: Stable event rates2. **Poisson Regression** (Section 2)   - Counts as function of predictors   - Log-link ensures positive rates   - Best for: Modeling rate changes with covariates3. **Negative Binomial Model** (Section 3)   - Overdispersed counts (variance > mean)   - Has extra dispersion parameter   - Best for: Highly variable count data4. **Model Comparison** (Section 4)   - LOO-CV for model selection   - Posterior predictive checks   - Best for: Choosing between competing models---

## Setup and ConfigurationWe'll use:- **PyMC**: Probabilistic programming framework- **ArviZ**: Bayesian visualization and diagnostics- **NumPy**: Data generation and manipulation- **Matplotlib/Seaborn**: Plotting

In [ ]:
# env option: asper_pymcimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport pymc as pmimport arviz as az# Set random seed for reproducibilityRANDOM_SEED = 42np.random.seed(RANDOM_SEED)rng = np.random.default_rng(RANDOM_SEED)# Configure plotting styleaz.style.use("arviz-darkgrid")print(f"Running on PyMC v{pm.__version__}")

---## Section 1: Basic Poisson Model### What is the Poisson Distribution?The **Poisson distribution** models the number of events occurring in a fixed interval of time or space. It's defined by a single parameter **λ** (lambda), which represents both the **mean** and **variance** of the distribution.**Mathematical form:**```P(X = k) = (λᵏ × e⁻λ) / k!```Where:- `X` = count (0, 1, 2, ...)- `k` = specific count value- `λ` = rate parameter (mean number of events)- `e` = Euler's number (≈ 2.718)**Key Properties:**- **Mean = λ**- **Variance = λ** (mean equals variance!)- **Support**: X ∈ {0, 1, 2, 3, ...}### When to Use Poisson?The Poisson distribution applies when:1. Events occur independently2. Events occur at a constant average rate3. Two events cannot occur at exactly the same instant4. The variance is approximately equal to the mean**Real-world examples:**- Number of emails received per hour- Number of mutations in a DNA sequence- Number of goals in a soccer match- Number of customers entering a store per day### GoalWe'll:1. Generate synthetic Poisson data with known λ = 52. Build a Bayesian model to estimate λ3. Verify the posterior contains the true value---### Step 1.1: Generate Synthetic Poisson DataLet's create 100 observations from a Poisson distribution with rate λ = 5.

In [ ]:
# True rate parametertrue_rate = 5# Generate 100 Poisson-distributed countsx = np.random.poisson(lam=true_rate, size=100)# Visualize the distributionplt.figure(figsize=(10, 4))plt.hist(x, bins=range(0, max(x)+2), alpha=0.7, edgecolor='black', density=True)plt.axvline(true_rate, color='red', linestyle='--', linewidth=2, label=f'True rate (λ={true_rate})')plt.axvline(x.mean(), color='blue', linestyle='--', linewidth=2, label=f'Sample mean ({x.mean():.2f})')plt.xlabel('Count')plt.ylabel('Probability Density')plt.title('Observed Poisson Data (n=100)')plt.legend()plt.grid(alpha=0.3)plt.show()print(f"First 10 observations: {x[:10]}")print(f"Sample mean: {x.mean():.2f}")print(f"Sample variance: {x.var():.2f}")print(f"Mean/Variance ratio: {x.mean() / x.var():.2f} (should be ≈1 for Poisson)")

**Observations:**- The histogram shows the distribution of our count data- Red dashed line: true rate (λ=5) used to generate data- Blue dashed line: sample mean (should be close to 5)- For Poisson data, mean ≈ variance (ratio ≈ 1)---### Step 1.2: Bayesian Poisson ModelNow we'll build a Bayesian model to **infer** the rate parameter λ from the observed counts.**Model Specification:****Prior:**```λ ~ Exponential(1)```- Weakly informative: allows λ to range widely but favors smaller values- Exponential prior has mean = 1, so we're being conservative**Likelihood:**```X ~ Poisson(λ)```- Each observation is Poisson-distributed with rate λ**Goal:** Compute the posterior distribution P(λ | data)

In [ ]:
# Define the Bayesian modelwith pm.Model() as poisson_model:    # Prior for rate parameter    # Exponential(1) has mean=1, allows flexibility for data to drive posterior    rate = pm.Exponential("rate", 1.0)        # Likelihood: observed counts follow Poisson distribution    counts = pm.Poisson("counts", mu=rate, observed=x)# Visualize model structurepm.model_to_graphviz(poisson_model)

**Understanding the Model:**- **rate**: The parameter we want to learn (λ)- **counts**: Our observed data (100 count values)- **Exponential(1) prior**: Allows rate to be any positive value, with mean=1---### Step 1.3: Sample from the PosteriorWe use **NUTS (No-U-Turn Sampler)** to draw samples from the posterior distribution.

In [ ]:
# Sample from posteriorwith poisson_model:    trace_poisson = pm.sample(1000, tune=1000, random_seed=RANDOM_SEED, chains=4)# Display sampling summaryaz.summary(trace_poisson)

**Interpreting Results:**| Metric | Value | Interpretation ||--------|-------|----------------|| **mean** | ~5.0 | Posterior mean estimate of λ || **sd** | ~0.2 | Posterior standard deviation (uncertainty) || **hdi_3%** | ~4.6 | Lower bound of 94% credible interval || **hdi_97%** | ~5.4 | Upper bound of 94% credible interval || **r_hat** | 1.0 | Convergence diagnostic (< 1.01 is good) || **ess_bulk** | >1000 | Effective sample size (>100 is adequate) |✅ **Success!** The posterior mean is very close to the true rate (5), and the true value falls within the 94% HDI.---### Step 1.4: Visualize Posterior Distribution

In [ ]:
# Create comprehensive visualizationfig, axes = plt.subplots(1, 3, figsize=(15, 4))# 1. Trace plot (left)az.plot_trace(trace_poisson, axes=axes[:2], combined=True)axes[0].set_title('Posterior Distribution of λ')axes[1].set_title('MCMC Trace')# 2. Forest plot (right)az.plot_forest(trace_poisson, ax=axes[2], hdi_prob=0.94)axes[2].axvline(true_rate, color='red', linestyle='--', linewidth=2, label='True λ')axes[2].set_title('94% Credible Interval')axes[2].legend()plt.tight_layout()plt.show()

**What to Look For:**1. **Posterior Distribution (left)**:    - Bell-shaped curve centered near true value   - Shows our uncertainty about λ2. **MCMC Trace (middle)**:   - Should look like "fuzzy caterpillar"   - Stable, well-mixed chains indicate good sampling3. **Forest Plot (right)**:   - Point estimate with uncertainty interval   - Red line shows true value falls within credible interval---

## Section 2: Poisson Regression### What is Poisson Regression?**Poisson regression** models count outcomes as a function of predictor variables. Instead of assuming a constant rate λ, we allow λ to vary based on covariates.**Mathematical Form:**```log(λᵢ) = β₀ + β₁X₁ᵢ + β₂X₂ᵢ + ...Yᵢ ~ Poisson(λᵢ)```**Key Components:**- **Log link function**: Ensures λᵢ > 0 (counts can't be negative)- **Linear predictor**: β₀ + β₁X₁ + ... (can be any real number)- **Exponential relationship**: λᵢ = exp(β₀ + β₁X₁ + ...)### Interpreting CoefficientsSince we use a log link:- **β₁** = change in log(λ) per 1-unit increase in X₁- **exp(β₁)** = multiplicative effect on λ**Example:** If β₁ = 0.5:- 1-unit increase in X₁ multiplies the rate by exp(0.5) ≈ 1.65 (65% increase)If β₁ = -0.3:- 1-unit increase in X₁ multiplies the rate by exp(-0.3) ≈ 0.74 (26% decrease)### When to Use Poisson Regression?Use when:- Your outcome is count data- You want to model how rates change with predictors- Mean and variance are approximately equal- Observations are independent**Real-world examples:**- Hospital admissions as a function of air pollution- Website clicks as a function of advertising spend- Disease cases as a function of vaccination rate---### Step 2.1: Generate Regression DataLet's simulate count data where the rate depends on a predictor X.

In [ ]:
# Set parameters for data generationnp.random.seed(123)n_obs = 100# Generate predictor variableX = np.random.normal(0, 1, n_obs)# True parameterstrue_alpha = 2.0   # Intercepttrue_beta = 0.5    # Slope# Generate rates using log-linktrue_lambda = np.exp(true_alpha + true_beta * X)# Generate Poisson countsY = np.random.poisson(true_lambda)# Visualize the relationshipfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Left: Y vs X with true rate curveaxes[0].scatter(X, Y, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)X_sorted = np.sort(X)axes[0].plot(X_sorted, np.exp(true_alpha + true_beta * X_sorted),              'r-', linewidth=2, label='True rate: λ = exp(2.0 + 0.5X)')axes[0].set_xlabel('Predictor (X)')axes[0].set_ylabel('Count (Y)')axes[0].set_title('Observed Counts vs Predictor')axes[0].legend()axes[0].grid(alpha=0.3)# Right: Distribution of Yaxes[1].hist(Y, bins=range(0, max(Y)+2), alpha=0.7, edgecolor='black')axes[1].set_xlabel('Count (Y)')axes[1].set_ylabel('Frequency')axes[1].set_title(f'Distribution of Observed Counts (mean={Y.mean():.1f})')axes[1].grid(alpha=0.3)plt.tight_layout()plt.show()print(f"True intercept (α): {true_alpha}")print(f"True slope (β): {true_beta}")print(f"Implied rate when X=0: exp({true_alpha}) = {np.exp(true_alpha):.2f}")print(f"Rate multiplier per unit X: exp({true_beta}) = {np.exp(true_beta):.2f}")print(f"\nObserved Y - mean: {Y.mean():.2f}, variance: {Y.var():.2f}")

**Observations:**- Left plot shows counts increasing exponentially with X (due to log-link)- Right plot shows the distribution of counts is right-skewed- Red line represents the true rate curve: λ = exp(2.0 + 0.5X)- When X increases by 1, the rate multiplies by exp(0.5) ≈ 1.65---### Step 2.2: Build Poisson Regression ModelNow we'll fit a Bayesian Poisson regression to recover the parameters.

In [ ]:
# Define Poisson regression modelwith pm.Model() as poisson_reg_model:    # Priors for regression coefficients    alpha = pm.Normal('alpha', mu=0, sigma=10)  # Intercept    beta = pm.Normal('beta', mu=0, sigma=10)    # Slope        # Linear predictor (on log scale)    eta = alpha + beta * X        # Expected count (inverse link: exp)    lambda_ = pm.math.exp(eta)        # Likelihood    Y_obs = pm.Poisson('Y_obs', mu=lambda_, observed=Y)# Visualize modelpm.model_to_graphviz(poisson_reg_model)

**Model Structure:**- **α, β**: Regression coefficients with weakly informative Normal priors- **η**: Linear predictor (can be any real number)- **λ**: Expected count (always positive due to exp link)- **Y**: Observed counts following Poisson distribution---### Step 2.3: Sample and Analyze

In [ ]:
# Sample from posteriorwith poisson_reg_model:    trace_reg = pm.sample(1000, tune=1000, random_seed=RANDOM_SEED, chains=4)# Display resultsprint("\n" + "="*60)print("POSTERIOR SUMMARY")print("="*60)summary = az.summary(trace_reg, hdi_prob=0.94)print(summary)# Calculate posterior meansalpha_post = trace_reg.posterior['alpha'].values.mean()beta_post = trace_reg.posterior['beta'].values.mean()print(f"\n{'='*60}")print("COMPARISON: TRUE vs POSTERIOR")print("="*60)print(f"Intercept (α):  True={true_alpha:.3f}, Posterior={alpha_post:.3f}")print(f"Slope (β):      True={true_beta:.3f}, Posterior={beta_post:.3f}")print(f"\nRate at X=0:   True={np.exp(true_alpha):.3f}, Posterior={np.exp(alpha_post):.3f}")print(f"Rate multiplier: True={np.exp(true_beta):.3f}, Posterior={np.exp(beta_post):.3f}")

**Interpretation:**✅ The posterior estimates closely match the true parameters!**Key insights:**- **exp(α)** gives the baseline rate when all predictors = 0- **exp(β)** gives the multiplicative effect of a 1-unit increase in X- If β = 0.5, each unit increase in X multiplies the rate by ~1.65x---### Step 2.4: Visualize Results

In [ ]:
# Create visualization gridfig = plt.figure(figsize=(16, 10))gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)# 1. Trace plotsax1 = fig.add_subplot(gs[0, :])az.plot_trace(trace_reg, compact=False, axes=[[ax1]])# 2. Posterior distributionsax2 = fig.add_subplot(gs[1, 0])az.plot_posterior(trace_reg, var_names=['alpha'], ref_val=true_alpha, ax=ax2, hdi_prob=0.94)ax2.set_title('Posterior: Intercept (α)')ax3 = fig.add_subplot(gs[1, 1])az.plot_posterior(trace_reg, var_names=['beta'], ref_val=true_beta, ax=ax3, hdi_prob=0.94)ax3.set_title('Posterior: Slope (β)')# 3. Fitted regression curve with uncertaintyax4 = fig.add_subplot(gs[2, :])# Extract posterior samplesposterior_samples = az.extract(trace_reg, num_samples=100)alpha_samples = posterior_samples['alpha'].valuesbeta_samples = posterior_samples['beta'].values# Plot posterior prediction bandsX_grid = np.linspace(X.min(), X.max(), 100)for i in range(len(alpha_samples)):    lambda_pred = np.exp(alpha_samples[i] + beta_samples[i] * X_grid)    ax4.plot(X_grid, lambda_pred, 'gray', alpha=0.05, zorder=1)# Plot observed dataax4.scatter(X, Y, alpha=0.6, s=50, c='blue', edgecolors='black', linewidth=0.5,             label='Observed data', zorder=3)# Plot true curvelambda_true = np.exp(true_alpha + true_beta * X_grid)ax4.plot(X_grid, lambda_true, 'r-', linewidth=2.5, label='True rate', zorder=4)# Plot posterior mean curvelambda_post_mean = np.exp(alpha_post + beta_post * X_grid)ax4.plot(X_grid, lambda_post_mean, 'g--', linewidth=2, label='Posterior mean', zorder=4)ax4.set_xlabel('Predictor (X)', fontsize=12)ax4.set_ylabel('Expected Count (λ)', fontsize=12)ax4.set_title('Poisson Regression: Data, True Model, and Posterior Fits', fontsize=13)ax4.legend(loc='upper left', fontsize=11)ax4.grid(alpha=0.3)plt.show()

**What We See:**1. **Top**: Trace plots show good MCMC convergence (fuzzy caterpillars)2. **Middle**: Posterior distributions for α and β, with red lines showing true values3. **Bottom**:    - Blue points: Observed data   - Red line: True rate curve   - Green dashed: Posterior mean estimate   - Gray lines: 100 draws from posterior (shows uncertainty)✅ The model successfully recovered the relationship between X and the count rate!---

## Section 3: Negative Binomial Model (Handling Overdispersion)### What is Overdispersion?**Overdispersion** occurs when the observed variance exceeds what the model predicts. For Poisson:- Poisson assumes: Variance = Mean- In reality: Often Variance > Mean (overdispersion)**Causes of overdispersion:**1. Unobserved heterogeneity (missing predictors)2. Clustering or correlation in data3. Excess zeros4. Contagion (one event triggers others)**Problem:** If we ignore overdispersion:- Standard errors are too small- Confidence intervals too narrow- Inflated Type I error rates### The Negative Binomial DistributionThe **Negative Binomial (NB)** distribution adds a dispersion parameter **α** (or **φ**):**Mathematical form (NB2 parameterization):**```Mean = μVariance = μ + μ²/α```Where:- **μ** = mean (like Poisson)- **α** = dispersion parameter- As α → ∞, NB → Poisson (variance = mean)- As α → 0, high overdispersion**Interpretation:**- Small α: High overdispersion (variance >> mean)- Large α: Low overdispersion (closer to Poisson)---### Step 3.1: Generate Overdispersed DataLet's create count data with variance > mean to demonstrate overdispersion.

In [ ]:
# Generate overdispersed count datanp.random.seed(456)n_obs = 100# Generate predictorX_nb = np.random.normal(0, 1, n_obs)# True parameterstrue_alpha_nb = 1.5true_beta_nb = 0.3true_dispersion = 2.0  # Controls overdispersion (smaller = more dispersion)# Expected countsmu_nb = np.exp(true_alpha_nb + true_beta_nb * X_nb)# Generate negative binomial counts# NB parameterization: n=alpha, p=alpha/(alpha+mu)Y_nb = np.random.negative_binomial(    n=true_dispersion,     p=true_dispersion / (true_dispersion + mu_nb))# For comparison, generate Poisson data with same meanY_poisson_comp = np.random.poisson(mu_nb)# Compare distributionsfig, axes = plt.subplots(1, 3, figsize=(16, 5))# Plot 1: NB data vs Xaxes[0].scatter(X_nb, Y_nb, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)X_sorted = np.sort(X_nb)mu_sorted = np.exp(true_alpha_nb + true_beta_nb * X_sorted)axes[0].plot(X_sorted, mu_sorted, 'r-', linewidth=2, label='Expected mean')axes[0].set_xlabel('Predictor (X)')axes[0].set_ylabel('Count (Y)')axes[0].set_title('Overdispersed Count Data (Negative Binomial)')axes[0].legend()axes[0].grid(alpha=0.3)# Plot 2: Compare NB vs Poisson distributionsaxes[1].hist(Y_nb, bins=range(0, max(Y_nb)+2), alpha=0.5, label='Negative Binomial', edgecolor='black')axes[1].hist(Y_poisson_comp, bins=range(0, max(Y_poisson_comp)+2), alpha=0.5, label='Poisson (same mean)', edgecolor='black')axes[1].set_xlabel('Count')axes[1].set_ylabel('Frequency')axes[1].set_title('NB vs Poisson: Same Mean, Different Variance')axes[1].legend()axes[1].grid(alpha=0.3)# Plot 3: Mean-Variance relationshipaxes[2].scatter(Y_nb.mean(), Y_nb.var(), s=100, c='blue', edgecolors='black', linewidth=2, label='NB data', zorder=3)axes[2].scatter(Y_poisson_comp.mean(), Y_poisson_comp.var(), s=100, c='red', marker='s', edgecolors='black', linewidth=2, label='Poisson data', zorder=3)axes[2].plot([0, max(Y_nb.mean(), Y_poisson_comp.mean())],              [0, max(Y_nb.mean(), Y_poisson_comp.mean())],              'k--', alpha=0.5, label='Variance = Mean line')axes[2].set_xlabel('Mean')axes[2].set_ylabel('Variance')axes[2].set_title('Mean-Variance Relationship')axes[2].legend()axes[2].grid(alpha=0.3)plt.tight_layout()plt.show()print("="*60)print("COMPARING NB vs POISSON DATA")print("="*60)print(f"\nNegative Binomial:")print(f"  Mean: {Y_nb.mean():.2f}")print(f"  Variance: {Y_nb.var():.2f}")print(f"  Variance/Mean ratio: {Y_nb.var()/Y_nb.mean():.2f}")print(f"\nPoisson (same mean):")print(f"  Mean: {Y_poisson_comp.mean():.2f}")print(f"  Variance: {Y_poisson_comp.var():.2f}")print(f"  Variance/Mean ratio: {Y_poisson_comp.var()/Y_poisson_comp.mean():.2f}")print(f"\n⚠️ Overdispersion detected! NB variance/mean = {Y_nb.var()/Y_nb.mean():.2f} > 1")print("   This violates Poisson assumption (variance = mean)")

**Key Observations:**1. **Left plot**: Overdispersed count data - notice the wide spread around the mean curve2. **Middle plot**: NB distribution (blue) has heavier tail than Poisson (orange)3. **Right plot**: NB data point is **above** the diagonal line (variance > mean)**This is overdispersion!** The Poisson model would underestimate the variance.---### Step 3.2: Fit Both Models and CompareWe'll fit both Poisson and Negative Binomial models to see which fits better.

In [ ]:
# Model 1: Poisson (misspecified - ignores overdispersion)with pm.Model() as poisson_wrong_model:    alpha = pm.Normal('alpha', mu=0, sigma=10)    beta = pm.Normal('beta', mu=0, sigma=10)        mu = pm.math.exp(alpha + beta * X_nb)        Y_obs = pm.Poisson('Y_obs', mu=mu, observed=Y_nb)        trace_pois_wrong = pm.sample(1000, tune=1000, random_seed=RANDOM_SEED)print("\n" + "="*60)print("POISSON MODEL (Misspecified for overdispersed data)")print("="*60)print(az.summary(trace_pois_wrong, hdi_prob=0.94))# Model 2: Negative Binomial (correctly specified)with pm.Model() as nb_model:    alpha = pm.Normal('alpha', mu=0, sigma=10)    beta = pm.Normal('beta', mu=0, sigma=10)        # Dispersion parameter (must be positive)    # Gamma prior is common for dispersion    alpha_disp = pm.Gamma('alpha_disp', mu=2, sigma=2)        mu = pm.math.exp(alpha + beta * X_nb)        # Negative Binomial likelihood    Y_obs = pm.NegativeBinomial('Y_obs', mu=mu, alpha=alpha_disp, observed=Y_nb)        trace_nb = pm.sample(1000, tune=1000, random_seed=RANDOM_SEED)print("\n" + "="*60)print("NEGATIVE BINOMIAL MODEL (Correct for overdispersed data)")print("="*60)print(az.summary(trace_nb, hdi_prob=0.94))

---### Step 3.3: Model ComparisonLet's compare the models to see which one fits better.

In [ ]:
# Compare parameter estimatesalpha_nb_post = trace_nb.posterior['alpha'].values.mean()beta_nb_post = trace_nb.posterior['beta'].values.mean()disp_nb_post = trace_nb.posterior['alpha_disp'].values.mean()alpha_pois_post = trace_pois_wrong.posterior['alpha'].values.mean()beta_pois_post = trace_pois_wrong.posterior['beta'].values.mean()print("="*70)print("PARAMETER COMPARISON: TRUE vs POISSON vs NEGATIVE BINOMIAL")print("="*70)print(f"{'Parameter':<20} {'True':<12} {'Poisson':<15} {'NB':<15}")print("-"*70)print(f"{'Intercept (α)':<20} {true_alpha_nb:<12.3f} {alpha_pois_post:<15.3f} {alpha_nb_post:<15.3f}")print(f"{'Slope (β)':<20} {true_beta_nb:<12.3f} {beta_pois_post:<15.3f} {beta_nb_post:<15.3f}")print(f"{'Dispersion':<20} {true_dispersion:<12.3f} {'N/A':<15} {disp_nb_post:<15.3f}")print()print(f"Dispersion interpretation: α = {disp_nb_post:.2f}")print(f"  -> Variance = μ + μ²/{disp_nb_post:.2f}")print(f"  -> Lower α = more overdispersion")# Visualize posteriorsfig, axes = plt.subplots(2, 2, figsize=(14, 10))# Row 1: Interceptsaz.plot_posterior(trace_pois_wrong, var_names=['alpha'], ref_val=true_alpha_nb,                   ax=axes[0, 0], hdi_prob=0.94)axes[0, 0].set_title('Poisson Model: Intercept (α)')az.plot_posterior(trace_nb, var_names=['alpha'], ref_val=true_alpha_nb,                   ax=axes[0, 1], hdi_prob=0.94)axes[0, 1].set_title('NB Model: Intercept (α)')# Row 2: Slopesaz.plot_posterior(trace_pois_wrong, var_names=['beta'], ref_val=true_beta_nb,                   ax=axes[1, 0], hdi_prob=0.94)axes[1, 0].set_title('Poisson Model: Slope (β)')az.plot_posterior(trace_nb, var_names=['beta'], ref_val=true_beta_nb,                   ax=axes[1, 1], hdi_prob=0.94)axes[1, 1].set_title('NB Model: Slope (β)')plt.tight_layout()plt.show()# Plot dispersion parameterfig, ax = plt.subplots(1, 1, figsize=(8, 5))az.plot_posterior(trace_nb, var_names=['alpha_disp'], ref_val=true_dispersion,                   ax=ax, hdi_prob=0.94)ax.set_title('Negative Binomial: Dispersion Parameter (α)')plt.show()

**Observations:**- Both models estimate α and β reasonably well- The NB model additionally estimates the dispersion parameter- Red lines show true parameter values fall within credible intervals---

## Section 4: Formal Model Comparison### Model Selection with LOO-CVWe'll use **Leave-One-Out Cross-Validation (LOO-CV)** to formally compare models:- **LOO**: Estimates out-of-sample predictive accuracy- **ELPD**: Expected Log Predictive Density (higher is better)- **WAIC**: Widely Applicable Information Criterion (lower is better)**How to interpret:**- **dse**: Standard error of the difference- If |difference| > 2×dse, models differ significantly- **p_loo**: Effective number of parameters- **Weight**: Relative model probability

In [ ]:
# Compute LOO-CV for both models# Add trace to InferenceData with model nameidata_pois = az.from_pymc(trace_pois_wrong, model=poisson_wrong_model)idata_nb = az.from_pymc(trace_nb, model=nb_model)# Compute LOO for each modelloo_pois = az.loo(idata_pois)loo_nb = az.loo(idata_nb)print("="*70)print("LOO-CV COMPARISON")print("="*70)print("\nPoisson Model:")print(loo_pois)print("\nNegative Binomial Model:")print(loo_nb)# Compare modelscomparison = az.compare({'Poisson': idata_pois, 'Negative Binomial': idata_nb})print("\n" + "="*70)print("MODEL COMPARISON SUMMARY")print("="*70)print(comparison)# Visualize comparisonaz.plot_compare(comparison, insample_dev=False, plot_standard_error=True)plt.title('Model Comparison: LOO-CV')plt.tight_layout()plt.show()# Interpretationbest_model = comparison.index[0]elpd_diff = comparison.loc['Poisson', 'elpd_diff'] if best_model == 'Negative Binomial' else 0se_diff = comparison.loc['Poisson', 'dse'] if best_model == 'Negative Binomial' else 0print("\n" + "="*70)print("INTERPRETATION")print("="*70)if abs(elpd_diff) > 2 * se_diff and se_diff > 0:    print(f"✅ {best_model} is significantly better (difference > 2×SE)")else:    print("⚠️ Models are not significantly different")print(f"\nELPD difference: {abs(elpd_diff):.2f} ± {se_diff:.2f}")print("\nFor overdispersed data, Negative Binomial typically performs better!")

**Understanding the Results:**- **rank**: 0 is best model- **elpd_loo**: Expected log predictive density (higher is better)- **p_loo**: Effective number of parameters- **weight**: Bayesian model averaging weight- **elpd_diff**: Difference from best model- **dse**: Standard error of the difference✅ **The Negative Binomial model should win!** It correctly accounts for overdispersion.---

## Section 5: Posterior Predictive Checks (PPC)### What are Posterior Predictive Checks?**PPC** validates our model by:1. Drawing parameter values from the posterior2. Generating new "replicated" data using those parameters3. Comparing replicated data to observed data**If the model is good:** Replicated data should look like observed data**What to check:**- Overall distribution shape- Mean and variance- Presence of outliers- Tail behavior---### Step 5.1: Generate Posterior Predictive Samples

In [ ]:
# Generate posterior predictive samples for both modelswith poisson_wrong_model:    ppc_pois = pm.sample_posterior_predictive(trace_pois_wrong, random_seed=RANDOM_SEED)with nb_model:    ppc_nb = pm.sample_posterior_predictive(trace_nb, random_seed=RANDOM_SEED)print("✅ Generated posterior predictive samples for both models")

---### Step 5.2: Visualize Posterior Predictive Distributions

In [ ]:
# Create comprehensive PPC visualizationfig, axes = plt.subplots(2, 2, figsize=(15, 10))# 1. Poisson PPC - KDEaz.plot_ppc(ppc_pois, num_pp_samples=100, ax=axes[0, 0], kind='kde')axes[0, 0].set_title('Poisson Model: PPC (KDE)', fontsize=12)axes[0, 0].set_xlabel('Count')# 2. NB PPC - KDEaz.plot_ppc(ppc_nb, num_pp_samples=100, ax=axes[0, 1], kind='kde')axes[0, 1].set_title('Negative Binomial Model: PPC (KDE)', fontsize=12)axes[0, 1].set_xlabel('Count')# 3. Poisson PPC - Cumulativeaz.plot_ppc(ppc_pois, num_pp_samples=100, ax=axes[1, 0], kind='cumulative')axes[1, 0].set_title('Poisson Model: PPC (Cumulative)', fontsize=12)axes[1, 0].set_xlabel('Count')# 4. NB PPC - Cumulative  az.plot_ppc(ppc_nb, num_pp_samples=100, ax=axes[1, 1], kind='cumulative')axes[1, 1].set_title('Negative Binomial Model: PPC (Cumulative)', fontsize=12)axes[1, 1].set_xlabel('Count')plt.tight_layout()plt.show()# Compute summary statistics for comparisonppc_pois_samples = ppc_pois.posterior_predictive['Y_obs'].valuesppc_nb_samples = ppc_nb.posterior_predictive['Y_obs'].valuesprint("="*70)print("POSTERIOR PREDICTIVE CHECK SUMMARY")print("="*70)print(f"\n{'Statistic':<20} {'Observed':<15} {'Poisson PPC':<18} {'NB PPC':<15}")print("-"*70)print(f"{'Mean':<20} {Y_nb.mean():<15.2f} {ppc_pois_samples.mean():<18.2f} {ppc_nb_samples.mean():<15.2f}")print(f"{'Variance':<20} {Y_nb.var():<15.2f} {ppc_pois_samples.var():<18.2f} {ppc_nb_samples.var():<15.2f}")print(f"{'Std Dev':<20} {Y_nb.std():<15.2f} {ppc_pois_samples.std():<18.2f} {ppc_nb_samples.std():<15.2f}")print(f"{'Min':<20} {Y_nb.min():<15.0f} {ppc_pois_samples.min():<18.0f} {ppc_nb_samples.min():<15.0f}")print(f"{'Max':<20} {Y_nb.max():<15.0f} {ppc_pois_samples.max():<18.0f} {ppc_nb_samples.max():<15.0f}")print("\n" + "="*70)print("ASSESSMENT")print("="*70)var_ratio_obs = Y_nb.var() / Y_nb.mean()var_ratio_pois = ppc_pois_samples.var() / ppc_pois_samples.mean()var_ratio_nb = ppc_nb_samples.var() / ppc_nb_samples.mean()print(f"Variance/Mean ratio:")print(f"  Observed data:          {var_ratio_obs:.2f}")print(f"  Poisson predictions:    {var_ratio_pois:.2f}")print(f"  NB predictions:         {var_ratio_nb:.2f}")print()if abs(var_ratio_nb - var_ratio_obs) < abs(var_ratio_pois - var_ratio_obs):    print("✅ Negative Binomial model better captures the observed overdispersion!")else:    print("⚠️ Poisson model may be adequate for this data")

**Interpretation:****Top row (KDE plots):**- Black line: Observed data distribution- Blue lines: 100 posterior predictive samples- Dark blue: Mean of posterior predictive**Bottom row (Cumulative plots):**- Same interpretation, but cumulative distribution- Easier to spot tail differences**Key insights:**1. **Poisson PPC**: May underestimate variance (tighter than observed)2. **NB PPC**: Better matches observed spread (wider, heavier tails)3. **Variance/Mean ratio**: NB predictions closer to observed ratio✅ **The NB model's predictions look more like the actual data!**---

## Section 6: Summary and Best Practices### What We LearnedIn this notebook, we covered:1. **Basic Poisson Model**   - Simplest count model with single rate parameter λ   - Assumes mean = variance   - Best for: Stable event rates without covariates2. **Poisson Regression**   - Models count rates as function of predictors   - Uses log-link to ensure positive rates   - Coefficients interpret as multiplicative effects (exp(β))   - Best for: Counts varying with explanatory variables3. **Negative Binomial Model**   - Extends Poisson with dispersion parameter α   - Handles overdispersion (variance > mean)   - More flexible than Poisson   - Best for: Highly variable count data4. **Model Comparison**   - LOO-CV for formal model selection   - Compare ELPD, WAIC, and model weights   - Best for: Choosing between competing models5. **Posterior Predictive Checks**   - Validate model by comparing predictions to data   - Check distributional assumptions   - Essential for model diagnosis---### Decision Tree: Which Model to Use?```Is your data COUNT data (non-negative integers)?│├─ NO → Use different model (linear, logistic, etc.)│└─ YES → Continue...    │    Do you have PREDICTORS?    │    ├─ NO → Use basic Poisson model    │   │    │   Check variance ≈ mean?    │   ├─ YES → Stick with Poisson    │   └─ NO (variance > mean) → Use Negative Binomial    │    └─ YES → Use Poisson Regression        │        Check variance ≈ mean (conditional on predictors)?        ├─ YES → Stick with Poisson Regression        └─ NO (overdispersion) → Use Negative Binomial Regression```---### Best Practices#### ✅ DO:1. **Always check for overdispersion**   - Compare sample variance to sample mean   - Plot mean-variance relationship   - Look at variance/mean ratio by group2. **Use informative priors when available**   - Incorporate domain knowledge   - Start with weakly informative priors   - Test sensitivity to prior choice3. **Run posterior predictive checks**   - Compare predictions to observed data   - Check multiple statistics (mean, variance, quantiles)   - Use both visual and numerical diagnostics4. **Check MCMC diagnostics**   - r_hat < 1.01 (convergence)   - ESS > 100 (effective sample size)   - No divergences   - Trace plots look like "fuzzy caterpillars"5. **Compare multiple models**   - Fit both Poisson and NB if uncertain   - Use LOO-CV for formal comparison   - Consider model weights for averaging6. **Visualize results clearly**   - Show uncertainty (credible intervals)   - Plot predicted rates with data   - Use posterior predictive distributions#### ❌ DON'T:1. **Don't ignore overdispersion**   - Poisson model will underestimate uncertainty   - Can lead to false confidence in results   - Always check variance/mean ratio2. **Don't forget the log-link interpretation**   - Coefficients are on log scale   - Use exp(β) for multiplicative effects   - Be careful with interaction terms3. **Don't skip model validation**   - Even if parameters "look right"   - Check predictions, not just parameter estimates   - Bad predictions = bad model4. **Don't use improper priors**   - Can lead to non-identifiable posteriors   - Use at least weakly informative priors   - Especially important for dispersion parameters5. **Don't rely on a single diagnostic**   - Use multiple checks (visual, numerical, predictive)   - No single metric tells the whole story---### Common Pitfalls and Solutions| Problem | Symptom | Solution ||---------|---------|----------|| **Overdispersion** | Variance >> Mean | Use Negative Binomial instead of Poisson || **Zero-inflation** | Too many zeros | Use Zero-Inflated Poisson/NB models || **Exposure variation** | Different observation periods | Use offset term: log(λ) = log(exposure) + β₀ + ... || **Convergence issues** | High r_hat, low ESS | Reparameterize, better priors, longer sampling || **Poor predictions** | PPC shows mismatch | Add predictors, change likelihood, check data quality || **Multicollinearity** | Wide credible intervals | Remove correlated predictors, use regularization |---### Key Formulas Reference**Poisson Distribution:**```Y ~ Poisson(λ)E[Y] = Var[Y] = λ```**Poisson Regression:**```log(λᵢ) = β₀ + β₁X₁ᵢ + β₂X₂ᵢ + ...Yᵢ ~ Poisson(λᵢ)```**Negative Binomial:**```Y ~ NegativeBinomial(μ, α)E[Y] = μVar[Y] = μ + μ²/α```**Overdispersion test:**```Variance/Mean ratio:  ≈ 1  → Poisson is appropriate  > 1  → Overdispersion (use NB)  < 1  → Underdispersion (rare)```---### Further Reading and Resources**PyMC Documentation:**- [Count Models Guide](https://www.pymc.io/projects/examples/en/latest/generalized_linear_models/GLM-count-regression.html)- [Model Comparison](https://www.pymc.io/projects/examples/en/latest/diagnostics_and_criticism/model_comparison.html)- [Posterior Predictive Checks](https://www.pymc.io/projects/examples/en/latest/diagnostics_and_criticism/posterior_predictive.html)**Statistical References:**- Gelman, A., et al. (2013). *Bayesian Data Analysis* (3rd ed.)- McElreath, R. (2020). *Statistical Rethinking* (2nd ed.)- Hilbe, J. M. (2011). *Negative Binomial Regression* (2nd ed.)**Related Topics:**- Zero-inflated models (for excess zeros)- Hierarchical count models (for grouped data)- Time series count models (for temporal correlation)- Spatial count models (for geographic data)---### Next StepsNow that you understand count models, explore:1. **Hierarchical models** (`variant_models/`)   - Multi-level count models   - Partial pooling for grouped data   - COVID variant modeling examples2. **Time series extensions**   - Autoregressive count models   - State-space models   - Trend and seasonality3. **Advanced count models**   - Zero-inflated Poisson/NB   - Hurdle models   - Conway-Maxwell-Poisson (for under/overdispersion)---## Questions or Issues?- Check the [PyMC documentation](https://www.pymc.io/)- See the main [README.md](../README.md) for troubleshooting- Review [CLAUDE.md](../CLAUDE.md) for project-specific guidance---**End of Notebook** Thank you for working through this tutorial! You now have the tools to analyze count data using Bayesian methods.